# NB08 — Literature novelty and evidence annotation
Implements the three-level novelty framework from the blueprint for candidate genes, using NCBI E-utilities
(`esearch`/`esummary` on the Gene and PubMed databases, no API key required at <= 3 requests/s).

**For each candidate:** (1) official symbol and aliases from NCBI Gene; (2) PubMed counts for the gene (symbol or alias,
title/abstract) combined with glioma terms, split into *any*, *prognosis/expression*, *functional* (knockdown, CRISPR,
overexpression, xenograft...) and *neural/synaptic context*; (3) the 20 most recent PMIDs with titles saved for **manual
reading**. Every query string and the run date are written to `09_Novelty_Assessment/query_log.csv`.

**Rules.** Candidates are the top genes by meta-analytic survival p-value (NB11), chosen **without** looking at
literature. A provisional label is derived from counts only and is **not** a novelty claim: the manual reading of
the saved PMIDs decides. A zero or low count never means "novel" (aliases, indexing lag, and abstract wording all
hide papers). Literature status is an annotation column in NB14 and never changes a candidate's tier.

**Smoke mode** runs the same code on three fixed self-test genes (EGFR, SNAP25, SYT1) to verify connectivity.

In [ ]:
import os, sys, time, json, re
ROOT = os.environ.get("GBM_ROOT", "/content/drive/MyDrive/GBM_MultiCohort_Research")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not in Colab; GBM_ROOT =", ROOT)
sys.path.insert(0, os.path.join(ROOT, "00_Project_Config"))
from gbm_common import *
import requests
EU = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
OUT = P("novelty"); N_CANDIDATES = 60
NCBI_EMAIL = ""      # optional: your e-mail, as NCBI asks for heavy users
RUN_DATE = dt.date.today().isoformat()
query_log = []

def eutils(endpoint, params, retries=4):
    params = {**params, "retmode": "json", "tool": "gbm_multicohort", **({"email": NCBI_EMAIL} if NCBI_EMAIL else {})}
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(EU + endpoint, params=params, timeout=40)
            if r.status_code == 429:
                time.sleep(2 * attempt); continue
            r.raise_for_status()
            time.sleep(0.4)
            return r.json()
        except Exception as exc:
            if attempt == retries:
                raise ActionableError(f"NCBI E-utilities request failed ({endpoint}): {exc}\nCheck internet access from the runtime "
                                      "and retry; results already saved are kept.")
            time.sleep(2 * attempt)

In [ ]:
DISEASE = '(glioblastoma[tiab] OR glioma[tiab] OR "high-grade glioma"[tiab] OR "grade 4 astrocytoma"[tiab] OR GBM[tiab])'
PROG = '(prognos*[tiab] OR survival[tiab] OR expression[tiab] OR biomarker[tiab])'
FUNC = '(knockdown[tiab] OR knockout[tiab] OR silencing[tiab] OR overexpression[tiab] OR inhibitor[tiab] OR xenograft[tiab] OR CRISPR[tiab] OR "in vivo"[tiab])'
NEURAL = '(synap*[tiab] OR neuron*[tiab] OR neural[tiab] OR axon*[tiab])'

def aliases_for(sym):
    j = eutils("esearch.fcgi", {"db": "gene", "term": f"{sym}[Gene Name] AND Homo sapiens[Organism]"})
    ids = j["esearchresult"].get("idlist", [])
    if not ids:
        return None, []
    s = eutils("esummary.fcgi", {"db": "gene", "id": ids[0]})["result"][ids[0]]
    if s.get("name", "").upper() != sym.upper():
        return None, []
    al = [a.strip() for a in s.get("otheraliases", "").split(",") if a.strip()]
    al = [a for a in al if len(a) >= 3 and not a.isdigit()][:8]    # short/numeric aliases are too noisy
    return s.get("description", ""), al

def count(term, label, gene):
    j = eutils("esearch.fcgi", {"db": "pubmed", "term": term, "retmax": 0})
    n = int(j["esearchresult"]["count"])
    query_log.append({"date": RUN_DATE, "gene": gene, "block": label, "query": term, "count": n})
    return n

def annotate(sym):
    desc, al = aliases_for(sym)
    if desc is None:
        return {"gene": sym, "gene_record_found": False}
    names = [sym] + al
    gene_clause = "(" + " OR ".join(f'"{n}"[tiab]' for n in names) + ")"
    base = f"{gene_clause} AND {DISEASE}"
    row = {"gene": sym, "gene_record_found": True, "description": desc, "aliases_used": ";".join(al),
           "n_any": count(base, "any", sym), "n_prognosis_expression": count(f"{base} AND {PROG}", "prognosis_expression", sym),
           "n_functional": count(f"{base} AND {FUNC}", "functional", sym), "n_neural_context": count(f"{base} AND {NEURAL}", "neural_context", sym)}
    j = eutils("esearch.fcgi", {"db": "pubmed", "term": base, "retmax": 20, "sort": "date"})
    pm = j["esearchresult"].get("idlist", [])
    titles = []
    if pm:
        sm = eutils("esummary.fcgi", {"db": "pubmed", "id": ",".join(pm)})["result"]
        titles = [{"gene": sym, "pmid": p, "year": sm[p].get("pubdate", "")[:4], "title": sm[p].get("title", ""), "journal": sm[p].get("source", "")} for p in pm]
    row["_titles"] = titles
    if row["n_functional"] >= 3:
        lab = "directly_studied_in_glioma (provisional)"
    elif row["n_any"] >= 3:
        lab = "partly_characterised: expression/prognosis mostly (provisional)"
    else:
        lab = "few_glioma_papers_found (provisional; NOT a novelty claim)"
    row["provisional_label"] = lab
    return row

In [ ]:
if SMOKE:
    genes, mode_note = ["EGFR", "SNAP25", "SYT1"], "SMOKE SELF-TEST (fixed genes; not candidates)"
else:
    f = P("clinical", "gene_cox_meta_M_genes.csv")
    require_files([f], "Run NB11 first.")
    genes = pd.read_csv(f).sort_values("meta_p").gene.head(N_CANDIDATES).tolist()
    mode_note = f"top {N_CANDIDATES} M genes by meta Cox p (literature not used for selection)"
print(mode_note, "|", len(genes), "genes | run date", RUN_DATE)

done = OUT / "literature_counts.csv"
prev = pd.read_csv(done) if done.exists() else pd.DataFrame()
have = set(prev.gene) if len(prev) else set()
rows, titles_all = ([] if not len(prev) else prev.to_dict("records")), []
tpath = OUT / "recent_pmids_for_manual_review.csv"
if tpath.exists():
    titles_all = pd.read_csv(tpath).to_dict("records")
for i, g in enumerate(genes, 1):
    if g in have:
        continue
    r = annotate(g)
    titles_all += r.pop("_titles", [])
    rows.append(r)
    if i % 10 == 0:
        pd.DataFrame(rows).to_csv(done, index=False); pd.DataFrame(titles_all).to_csv(tpath, index=False)
        print(f"  {i}/{len(genes)} genes done (checkpoint saved)")
lit = pd.DataFrame(rows); lit["mode"] = mode_note; lit["run_date"] = RUN_DATE
lit.to_csv(done, index=False); pd.DataFrame(titles_all).to_csv(tpath, index=False)
pd.DataFrame(query_log).to_csv(OUT / "query_log.csv", mode="a", header=not (OUT / "query_log.csv").exists(), index=False)
print(lit.drop(columns=[c for c in ["description", "aliases_used", "mode"] if c in lit.columns]).to_string(index=False))
manifest_update("novelty", "run", {"date": RUN_DATE, "mode": mode_note, "n_genes": len(lit), "queries_logged": len(query_log),
                                   "source": "NCBI E-utilities (Gene, PubMed)"})
if SMOKE:
    print("\nSMOKE self-test only: counts above are real PubMed counts for three known genes, not study results.")
print("\nNB08 complete. Manually read recent_pmids_for_manual_review.csv before making any novelty statement.")